In [1]:
import numpy as np





def sigmoid(x):
    return 1 / (1 + np.exp(-x))
def sigmoid_derivative(x):
    return x * (1 - x)



In [2]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def sigmoid_derivative(z):
    s = sigmoid(z)
    return s * (1 - s)

def softmax(z):
    exp_z = np.exp(z - np.max(z, axis=1, keepdims=True))  # for stability
    return exp_z / np.sum(exp_z, axis=1, keepdims=True)

def ann_numpy_classifier(X_train, y_train, X_test, y_test, epochs=1000, hidden_size=8, learning_rate=0.01):
    np.random.seed(42)

    input_size = X_train.shape[1]
    output_size = y_train.shape[1]

    # Weight and bias initialization
    W1 = np.random.randn(input_size, hidden_size)
    b1 = np.zeros((1, hidden_size))

    W2 = np.random.randn(hidden_size, output_size)
    b2 = np.zeros((1, output_size))

    # Training loop
    for epoch in range(epochs):
        # Forward pass
        Z1 = np.dot(X_train, W1) + b1
        A1 = sigmoid(Z1)

        Z2 = np.dot(A1, W2) + b2
        A2 = softmax(Z2)

        # Loss (cross-entropy)
        loss = -np.mean(np.sum(y_train * np.log(A2 + 1e-8), axis=1))

        # Backward pass
        dZ2 = A2 - y_train
        dW2 = np.dot(A1.T, dZ2)
        db2 = np.sum(dZ2, axis=0, keepdims=True)

        dA1 = np.dot(dZ2, W2.T)
        dZ1 = dA1 * sigmoid_derivative(Z1)
        dW1 = np.dot(X_train.T, dZ1)
        db1 = np.sum(dZ1, axis=0, keepdims=True)

        # Update weights
        W2 -= learning_rate * dW2
        b2 -= learning_rate * db2
        W1 -= learning_rate * dW1
        b1 -= learning_rate * db1

        if epoch % 100 == 0 or epoch == epochs - 1:
            print(f"Epoch {epoch}, Loss: {loss:.4f}")

    # ---- Prediction ----
    def predict(X):
        A1 = sigmoid(np.dot(X, W1) + b1)
        A2 = softmax(np.dot(A1, W2) + b2)
        return np.argmax(A2, axis=1)

    y_pred = predict(X_test)
    y_true = np.argmax(y_test, axis=1)

    print("\nClassification Report:")
    print(classification_report(y_true, y_pred))

    print("Confusion Matrix:")
    print(confusion_matrix(y_true, y_pred))

    return y_pred, (W1, b1, W2, b2)
